[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SEU_USUARIO/gr-notebooks/blob/main/aula-big-data/demonstracao-lakehouse-medalhao.ipynb)

# Demonstração Prática: Plataformas de Big Data e um Lakehouse com Arquitetura Medalhão

Este notebook é uma demonstração prática dos conceitos apresentados no notebook de conceitos básicos desta mesma pasta: ecossistema de ferramentas de Big Data, arquitetura Lakehouse e camadas Medalhão (Bronze, Silver, Gold).

Vamos em três etapas: primeiro, um comparativo objetivo das principais plataformas de Big Data/Lakehouse disponíveis no mercado hoje. Depois, um mini Lakehouse rodando localmente, com pandas, só para fixar a mecânica de Bronze → Silver → Gold sem depender de nenhuma conta na nuvem. Por fim, um passo a passo de como o mesmo padrão é implementado de verdade em duas das plataformas mais usadas atualmente: **Microsoft Fabric** e **Databricks**.

## Comparativo de Ambientes de Big Data no Mercado

Não existe "a" ferramenta de Big Data — existe um mercado de plataformas que resolvem o mesmo problema (armazenamento e processamento distribuído, em arquitetura de Lakehouse) com peças e nomes diferentes. A tabela abaixo compara cinco das plataformas mais relevantes hoje, nas mesmas dimensões que discutimos na teoria: formato de armazenamento, motor de processamento, orquestração, governança, cobrança, BI e streaming.

| Plataforma | Formato de tabela aberto | Motor de processamento | Orquestração nativa | Catálogo / Governança | Modelo de cobrança | BI nativo | Streaming |
|---|---|---|---|---|---|---|---|
| **Databricks** | Delta Lake (padrão); também lê/grava Iceberg via UniForm | Apache Spark + Photon (motor SQL vetorizado) | Lakeflow Jobs (antigo "Databricks Workflows"; no menu aparece como "Jobs & Pipelines") | Unity Catalog (metastore + controle de acesso + linhagem automática) | DBU (Databricks Unit), consumo por segundo | AI/BI Dashboards + Genie (perguntas em linguagem natural) | Structured Streaming (nativo do Spark) |
| **Microsoft Fabric** | Delta Lake (padronizado em todos os motores) | Apache Spark (Notebooks) + motor SQL proprietário (item Warehouse) | Data Factory → item "Data pipeline" | OneLake catalog (+ integração opcional com Microsoft Purview) | CU (Capacity Unit), via capacidades F2 a F2048 | Power BI nativo (modo Direct Lake, sem cópia dos dados) | Eventstream (parte do Real-Time Intelligence) |
| **Snowflake** | Formato proprietário; suporte nativo a Apache Iceberg via Polaris/Horizon Catalog | Motor SQL proprietário; Snowpark para código em estilo Spark | Snowflake Tasks (`CREATE TASK ... AFTER`, formam um DAG) | Horizon Catalog, construído sobre o Polaris Catalog (open source) | Créditos, em 3 medidores: computação, armazenamento e serviços de nuvem | Snowsight (dashboards nativos) + conectores para Tableau/Power BI/Looker | Snowpipe Streaming |
| **AWS** | Apache Iceberg (padrão atual, incl. S3 Tables gerenciado) + Hive legado | Spark (EMR/Glue) + motores proprietários (Redshift, Athena) | AWS Glue Workflows; Step Functions/Amazon MWAA para orquestração mais ampla | AWS Glue Data Catalog + AWS Lake Formation (controle de acesso) | Por serviço: DPU-hora (Glue), hora de instância (EMR/Redshift), GB armazenado (S3) | Amazon QuickSight | Amazon Kinesis / Amazon MSK (Kafka gerenciado) |
| **Google Cloud** | Apache Iceberg, via "Lakehouse for Apache Iceberg" (renomeado do BigLake em 2026) | BigQuery (motor Dremel, serverless) + Spark (Dataproc) | Cloud Composer (Apache Airflow gerenciado) + Dataform | Dataplex Universal Catalog | Slots (BigQuery) ou DCU (Dataproc), por segundo | Looker + BigQuery BI Engine | Dataflow (Apache Beam) + Pub/Sub |

> 📝 **Nota:** os valores de cobrança variam por região, edição/contrato e volume negociado — por isso a tabela mostra apenas a *unidade* de consumo de cada plataforma (DBU, CU, Crédito...), não valores em R\$/US\$. Os links para a documentação oficial de cada item estão na seção "Saiba Mais", ao final.

Um pouco de contexto por trás de cada linha:

- **Databricks** foi fundada em 2013 pelos criadores do Apache Spark. Foi lá que os pesquisadores Matei Zaharia e Ali Ghodsi formalizaram o termo **"Lakehouse"** — o mesmo paper de 2021 que vimos no notebook de conceitos básicos desta pasta.
- **Microsoft Fabric** unifica em um único SaaS produtos que antes eram separados (Power BI, Synapse Analytics, Data Factory), com o **OneLake** funcionando como um "lake" lógico único para todo o tenant. "Lakehouse" é, literalmente, um tipo de item que se cria dentro de um workspace do Fabric.
- **Snowflake** foi pioneira em separar armazenamento e computação em unidades independentes e escaláveis (os "warehouses" virtuais), e abriu o código do **Polaris Catalog** para tornar tabelas Iceberg interoperáveis entre motores de diferentes fornecedores.
- **AWS**: o Amazon S3 (2006) é anterior ao próprio conceito de Lakehouse e se tornou a base de armazenamento sobre a qual boa parte do ecossistema de formatos abertos (Iceberg, Delta, Hudi) foi construído — inclusive em outras nuvens.
- **Google Cloud**: o paper do Dremel (2010) é a base da arquitetura serverless do BigQuery, um dos primeiros motores SQL MPP totalmente serverless do mercado.

> 💡 **Dica:** repare que "orquestração nativa" e "processamento" quase sempre aparecem como produtos separados dentro da mesma plataforma (ex.: Glue Workflows ≠ EMR; Data Factory ≠ Spark no Fabric). Isso reflete a mesma separação entre as peças do ecossistema de Big Data que vimos na teoria — armazenamento, processamento e orquestração continuam sendo problemas distintos, mesmo quando uma única plataforma resolve todos eles.

## Colocando a Mão na Massa: um Mini Lakehouse Local

Antes de ir para a nuvem, vamos fixar a mecânica da arquitetura medalhão com uma versão mínima rodando localmente, com pandas, sobre os dados de `departamentos` que já conhecemos. A ideia não é reproduzir Delta Lake ou Spark — é isolar o padrão Bronze → Silver → Gold em três funções simples, para depois reconhecer exatamente as mesmas peças quando virmos a versão "de verdade" no Fabric e no Databricks.

In [ ]:
import pandas as pd
import glob
import os
import time
from datetime import datetime

### Bronze: ingestão crua

A camada Bronze lê todas as partições mensais da landing zone exatamente como estão, sem limpar nem deduplicar nada — só adiciona metadados de auditoria (de qual arquivo veio, quando foi ingerido).

In [ ]:
def etl_bronze_departamentos(
    caminho_landing='../data/landing/departamentos',
    caminho_bronze='../data/lakehouse/bronze',
):
    # lê todas as partições mensais, sem transformar nada
    caminhos_particoes = sorted(glob.glob(f'{caminho_landing}/dt=*/departamentos.csv'))

    partes = []
    for caminho in caminhos_particoes:
        df_particao = pd.read_csv(caminho)
        # metadados de auditoria: de onde veio e quando foi ingerido
        df_particao['_arquivo_origem'] = caminho
        df_particao['_timestamp_ingestao'] = datetime.now().isoformat(timespec='seconds')
        partes.append(df_particao)

    df_bronze = pd.concat(partes, ignore_index=True)

    os.makedirs(caminho_bronze, exist_ok=True)
    df_bronze.to_csv(f'{caminho_bronze}/departamentos.csv', index=False)

    print(f'[BRONZE] {len(df_bronze)} linhas gravadas a partir de {len(caminhos_particoes)} partições.')
    return df_bronze

### Silver: limpeza e deduplicação

A camada Silver lê o Bronze, corrige tipos, limpa espaços em branco e reduz cada departamento à sua versão mais recente.

In [ ]:
def etl_silver_departamentos(
    caminho_bronze='../data/lakehouse/bronze',
    caminho_silver='../data/lakehouse/silver',
):
    df_bronze = pd.read_csv(f'{caminho_bronze}/departamentos.csv')

    # tipagem explícita e limpeza de espaços em branco
    df_bronze['data_extracao'] = pd.to_datetime(df_bronze['data_extracao'])
    df_bronze['nome_departamento'] = df_bronze['nome_departamento'].str.strip()

    # mantém apenas o registro mais recente de cada departamento
    df_silver = (
        df_bronze
        .sort_values('data_extracao')
        .drop_duplicates(subset='departamento_id', keep='last')
        .drop(columns=['_arquivo_origem', '_timestamp_ingestao'])
        .reset_index(drop=True)
    )

    os.makedirs(caminho_silver, exist_ok=True)
    df_silver.to_csv(f'{caminho_silver}/departamentos.csv', index=False)

    print(f'[SILVER] {len(df_silver)} departamentos únicos após deduplicação.')
    return df_silver

> ⚠️ **Atenção:** para manter a demonstração simples, aqui ficamos apenas com a versão mais recente de cada departamento — o equivalente a uma SCD Tipo 1. Um pipeline de produção normalmente usaria SCD Tipo 2 (como vimos na teoria) para preservar o histórico completo de mudanças, em vez de sobrescrevê-lo.

### Gold: modelo pronto para consumo analítico

A camada Gold lê o Silver e produz uma tabela já agregada para responder a uma pergunta de negócio específica — nesse caso, uma pergunta de **analytics descritiva** simples: quantos departamentos subordinados cada área tem?

In [ ]:
def etl_gold_estrutura_organizacional(
    caminho_silver='../data/lakehouse/silver',
    caminho_gold='../data/lakehouse/gold',
):
    df_silver = pd.read_csv(f'{caminho_silver}/departamentos.csv')

    # conta quantos departamentos têm cada departamento_id como pai
    df_gold = (
        df_silver
        .dropna(subset=['departamento_pai_id'])
        .groupby('departamento_pai_id')
        .size()
        .reset_index(name='qtd_subdepartamentos')
        .rename(columns={'departamento_pai_id': 'departamento_id'})
        .merge(df_silver[['departamento_id', 'nome_departamento']], on='departamento_id', how='left')
        .sort_values('qtd_subdepartamentos', ascending=False)
    )

    os.makedirs(caminho_gold, exist_ok=True)
    df_gold.to_csv(f'{caminho_gold}/estrutura_organizacional.csv', index=False)

    print(f'[GOLD] tabela "estrutura_organizacional" gerada com {len(df_gold)} linhas.')
    return df_gold

### Orquestração: encadeando as três etapas

Uma orquestração de verdade (como veremos a seguir no Fabric e no Databricks) cuida de agendamento, novas tentativas em caso de falha, alertas e execução em paralelo quando possível. Aqui, uma versão mínima: apenas respeitar a ordem de dependência (Bronze antes de Silver, Silver antes de Gold) e registrar o resultado de cada etapa.

In [ ]:
# ORQUESTRAÇÃO (simplificada): executa as etapas em sequência, respeitando a dependência Bronze -> Silver -> Gold
etapas = [
    ('bronze', etl_bronze_departamentos),
    ('silver', etl_silver_departamentos),
    ('gold', etl_gold_estrutura_organizacional),
]

print('Iniciando pipeline "departamentos"\n' + '-' * 40)
for nome_etapa, funcao_etapa in etapas:
    inicio = time.time()
    funcao_etapa()
    duracao = time.time() - inicio
    print(f'  -> etapa "{nome_etapa}" concluída em {duracao:.2f}s')
print('-' * 40 + '\nPipeline concluído com sucesso.')

In [ ]:
# conferindo o resultado final da camada Gold
pd.read_csv('../data/lakehouse/gold/estrutura_organizacional.csv')

Repare que a tabela Gold já responde diretamente a uma pergunta de negócio ("Diretoria tem 6 departamentos subordinados, Vendas tem 1"), sem exigir nenhum `join` ou tratamento adicional de quem for consumi-la — exatamente o papel que a camada Gold deve cumprir.

## Passo a Passo: Levando o Mesmo Padrão para a Nuvem

O mini Lakehouse local que acabamos de construir tem exatamente as mesmas três peças que qualquer Lakehouse "de verdade": uma camada de ingestão crua, uma de limpeza e uma de consumo analítico, executadas em uma ordem de dependência fixa. A diferença é que, na nuvem, cada uma dessas peças é implementada por um produto gerenciado específico. Veja como o mesmo pipeline (Bronze → Silver → Gold, com 3 notebooks/scripts orquestrados) é montado no **Microsoft Fabric** e no **Databricks**.

### No Microsoft Fabric

**1. Criar o workspace e os itens Lakehouse**

Crie um [workspace do Fabric](https://learn.microsoft.com/en-us/fabric/fundamentals/create-workspaces) e, dentro dele, um item **Lakehouse** para cada camada: `lh_bronze`, `lh_silver` e `lh_gold`. Isso é feito em **Novo item → pesquisar "Lakehouse" → nomear → Criar**. A documentação oficial de arquitetura medalhão do Fabric recomenda exatamente esse padrão — um Lakehouse por camada, cada um com suas próprias áreas **Files** (arquivos brutos) e **Tables** (tabelas Delta), ambas apoiadas no OneLake.

> 💡 **Dica:** para uma demonstração rápida, um único Lakehouse com pastas separadas também funciona — mas separar por Lakehouse (ou até por workspace) facilita aplicar permissões diferentes por camada mais adiante.

**2. Ingerir os arquivos brutos no Bronze**

Em `lh_bronze`: **Obter dados → Carregar arquivos** (`Get data → Upload files`) para subir o CSV (`departamentos.csv`) na área **Files** — a "zona de pouso" para dados crus, em qualquer formato. Em seguida, clique com o botão direito no arquivo → **Carregar para Tabelas → Nova tabela** (`Load to Tables → New table`): o Fabric infere o schema e grava automaticamente uma **tabela Delta** na área **Tables**. Para volumes maiores, o Fabric também oferece **atalhos (shortcuts)** — ponteiros de acesso direto a dados no ADLS Gen2, S3 ou em outro Lakehouse, sem copiar nada — indicados justamente para a camada Bronze.

**3. Notebook Silver e notebook Gold**

Crie um notebook Fabric (PySpark) que lê a tabela Delta de `lh_bronze`, limpa e deduplica os dados, e grava o resultado como tabela em `lh_silver` — a mesma lógica da função `etl_silver_departamentos` que escrevemos acima, agora em Spark. Crie um segundo notebook que lê `lh_silver`, agrega e grava em `lh_gold`. Para acessar um Lakehouse diferente do notebook a partir de outro, use o utilitário `notebookutils` (nome atual; `mssparkutils` é o alias legado, ainda funcional) — inclusive `notebookutils.notebook.run()`/`runMultiple()` permitem chamar um notebook a partir de outro programaticamente.

**4. Orquestrar com um Data pipeline**

Crie um item **Data pipeline** (**Novo item → Data pipeline**, dentro do workload Data Factory). Adicione três **atividades do tipo Notebook** (`Notebook activity`), uma para cada etapa, e conecte-as em sequência pelas setas de dependência de sucesso (Bronze → Silver → Gold) — assim como fizemos com a lista `etapas` no exemplo local, mas agora com retentativas, alertas e paralelismo geridos pela própria plataforma. Configure um **gatilho de agendamento** (`Schedule`) para rodar, por exemplo, todo dia às 6h — ou um gatilho por evento, disparado quando um novo arquivo chega no armazenamento.

> 📝 **Nota:** o Fabric também oferece **materialized lake views**, uma forma declarativa de descrever as transformações Bronze/Silver/Gold em SQL, deixando que a própria plataforma calcule a ordem de execução e a estratégia de atualização — o equivalente, no Fabric, ao Lakeflow Declarative Pipelines do Databricks (próxima seção).

### No Databricks

**1. Organizar o Unity Catalog**

Dentro de um catálogo do **Unity Catalog**, crie três *schemas* (o equivalente a "bancos" dentro do catálogo): `bronze`, `silver` e `gold` — por exemplo, `meu_catalogo.bronze`, `meu_catalogo.silver`, `meu_catalogo.gold`. É exatamente esse padrão de nomenclatura que a documentação oficial usa em seus próprios exemplos de arquitetura medalhão.

**2. Ingerir os arquivos brutos em um Volume**

Dentro do schema `bronze`, crie um **Volume** (armazenamento de arquivos governado pelo Unity Catalog): no Catalog Explorer, **Create volume**, depois **Upload to this volume** para subir o CSV. Um notebook Bronze lê os arquivos desse volume e grava como **tabela Delta gerenciada** em `meu_catalogo.bronze.departamentos` — no Databricks, toda tabela criada sem especificar formato já nasce como tabela Delta.

**3. Notebook Silver e notebook Gold**

Um segundo notebook lê `meu_catalogo.bronze.departamentos`, aplica a mesma lógica de limpeza e deduplicação da função `etl_silver_departamentos`, e grava em `meu_catalogo.silver.departamentos`. Um terceiro notebook lê o Silver, agrega, e grava em `meu_catalogo.gold.estrutura_organizacional`. Para prototipar rapidamente, um notebook pode chamar outro com `%run ./caminho_do_notebook` (herda variáveis e funções) ou `dbutils.notebook.run(caminho, timeout, parametros)` (executa como um job à parte, com parâmetros) — mas, para produção, a própria documentação recomenda migrar esse encadeamento para a ferramenta de orquestração nativa.

**4. Orquestrar com Lakeflow Jobs**

No menu lateral **Jobs & Pipelines** (o nome do produto na documentação é **Lakeflow Jobs** — o sucessor direto do que era chamado de "Databricks Workflows"), crie um **Job** com três *tasks* do tipo Notebook, uma para cada etapa. Em cada task, defina **"Depends on"** apontando para a task anterior, formando a mesma cadeia Bronze → Silver → Gold. Configure um **trigger** de agendamento (cron) ou baseado em evento (chegada de um novo arquivo no armazenamento em nuvem).

> 📝 **Nota:** como alternativa mais declarativa, o Databricks oferece o **Lakeflow Declarative Pipelines** (o nome atual do que já foi chamado de Delta Live Tables / DLT): em vez de escrever três notebooks e orquestrá-los manualmente, você declara cada tabela (`@dp.table`) e a própria plataforma resolve o grafo de dependências e a atualização incremental. Para quem está começando, o **Lakeflow Designer** oferece ainda uma versão sem código para montar esse mesmo tipo de pipeline visualmente.

> 💡 **Dica:** a **Databricks Community Edition** foi descontinuada em janeiro de 2026 e substituída pela **Databricks Free Edition** — é por ela que vale a pena começar se você quiser reproduzir este passo a passo sem custo.

## Saiba Mais

**Arquitetura medalhão (documentação oficial):**
- [Medallion architecture (Databricks)](https://docs.databricks.com/aws/en/lakehouse/medallion)
- [Medallion Lakehouse Architecture (Microsoft Fabric)](https://learn.microsoft.com/en-us/fabric/onelake/onelake-medallion-lakehouse-architecture)
- [Tutorial: Lakehouse end-to-end (Microsoft Fabric)](https://learn.microsoft.com/en-us/fabric/data-engineering/tutorial-lakehouse-introduction)

**Databricks:**
- [Jobs & Pipelines / Lakeflow Jobs](https://docs.databricks.com/aws/en/jobs/)
- [Unity Catalog](https://docs.databricks.com/aws/en/data-governance/unity-catalog/)
- [Delta Lake quickstart](https://docs.databricks.com/aws/en/delta/tutorial)
- [Lakeflow Declarative Pipelines (Delta Live Tables)](https://docs.databricks.com/aws/en/ldp/concepts/where-is-dlt)
- [Databricks Free Edition](https://docs.databricks.com/aws/en/getting-started/free-edition)

**Microsoft Fabric:**
- [Notebook activity (Data Factory)](https://learn.microsoft.com/en-us/fabric/data-factory/notebook-activity)
- [Notebook utilities (notebookutils)](https://learn.microsoft.com/en-us/fabric/data-engineering/notebook-utilities)
- [Quickstart: criar um Lakehouse](https://learn.microsoft.com/en-us/fabric/onelake/quickstart-get-data)

**Outras plataformas:**
- [Introducing Polaris Catalog (Snowflake)](https://www.snowflake.com/en/blog/introducing-polaris-catalog/)
- [Amazon S3 Tables / Lake Formation (AWS)](https://docs.aws.amazon.com/lake-formation/latest/dg/create-s3-tables-catalog.html)
- [Lakehouse for Apache Iceberg (Google Cloud)](https://docs.cloud.google.com/lakehouse/docs/introduction)

## Finalizando

Nesta demonstração você saiu de um comparativo objetivo das principais plataformas de Big Data do mercado, passou por um mini Lakehouse rodando localmente com pandas — só para fixar a mecânica de Bronze, Silver e Gold — e chegou a um passo a passo real de como esse mesmo padrão é implementado no Microsoft Fabric e no Databricks, incluindo a orquestração do pipeline.

A peça mais importante para levar daqui: a arquitetura medalhão não é exclusiva de nenhuma ferramenta. É um padrão de organização de dados que qualquer plataforma de Lakehouse — gerenciada ou não — pode implementar, com nomes de produtos diferentes cumprindo exatamente os mesmos papéis.

Um abraço e até a próxima,

Walter.